# NB_SETUP_POI_ENTITY

Tạo và seed `lh_vv_silver.dbo.ref_dictionary_place_type` (insert-only), gồm `category_group`. `p_apply` mặc định `false`: in kế hoạch rồi thoát, không ghi.

Ghi 5 cạnh vào `slv_poi_entity_non_enrich` và retire 2 cạnh `slv_poi_entity`.


In [ ]:
p_apply = "false"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
# BEGIN_PLACE_TYPE_SEED
import re

SEED_SECTORS = (
    "accommodation",
    "f_n_b",
    "spa_n_wellness",
    "shopping_n_retail",
    "attraction",
    "vehicle_rental",
    "other",
)
SEED_SKIP = (
    "Partner Portal bổ sung",
    "Partner Portal Bổ sung",
    "Fallback",
    "Không có",
)
SEED_ROWS = [
    ('partner', 'Hotel', 'accommodation', 'Hotel', 'Khách sạn'),
    ('google', 'hotel', 'accommodation', 'Hotel', 'Khách sạn'),
    ('google', 'extended_stay_hotel', 'accommodation', 'Extended Stay Hotel', 'Lưu trú dài hạn'),
    ('google', 'inn', 'accommodation', 'Inn', 'Khách sạn nhỏ'),
    ('partner', 'Resort', 'accommodation', 'Resort', 'Khu nghỉ dưỡng'),
    ('google', 'resort_hotel', 'accommodation', 'Resort', 'Khu nghỉ dưỡng'),
    ('partner', 'Hostel', 'accommodation', 'Hostel', 'Hostel'),
    ('google', 'hostel', 'accommodation', 'Hostel', 'Hostel'),
    ('partner', 'Motel', 'accommodation', 'Motel', 'Motel'),
    ('google', 'motel', 'accommodation', 'Motel', 'Motel'),
    ('partner', 'Homestay', 'accommodation', 'Homestay', 'Homestay'),
    ('google', 'guest_house', 'accommodation', 'Homestay', 'Homestay'),
    ('google', 'private_guest_room', 'accommodation', 'Homestay', 'Homestay'),
    ('google', 'farmstay', 'accommodation', 'Farmstay', 'Farmstay'),
    ('partner', 'Campground/Glamping', 'accommodation', 'Campground', 'Khu cắm trại'),
    ('google', 'campground', 'accommodation', 'Campground', 'Khu cắm trại'),
    ('google', 'camping_cabin', 'accommodation', 'Camping Cabin', 'Nhà gỗ cắm trại'),
    ('google', 'bed_and_breakfast', 'accommodation', 'Bed & Breakfast', 'Bed & Breakfast'),
    ('google', 'cottage', 'accommodation', 'Cottage', 'Nhà gỗ'),
    ('google', 'mobile_home_park', 'accommodation', 'Mobile Home Park', 'Khu nhà lưu động'),
    ('google', 'lodging', 'accommodation', 'Accommodation', 'Lưu trú'),
    ('google', 'budget_japanese_inn', 'accommodation', 'Inn', 'Khách sạn nhỏ'),
    ('google', 'japanese_inn', 'accommodation', 'Inn', 'Khách sạn nhỏ'),
    ('partner', 'Vietnamese', 'f_n_b', 'Vietnamese restaurant', 'Nhà hàng Việt'),
    ('google', 'vietnamese_restaurant', 'f_n_b', 'Vietnamese restaurant', 'Nhà hàng Việt'),
    ('partner', 'Chinese', 'f_n_b', 'Chinese restaurant', 'Nhà hàng Trung'),
    ('google', 'chinese_restaurant', 'f_n_b', 'Chinese restaurant', 'Nhà hàng Trung'),
    ('partner', 'Japanese', 'f_n_b', 'Japanese restaurant', 'Nhà hàng Nhật'),
    ('google', 'japanese_restaurant', 'f_n_b', 'Japanese restaurant', 'Nhà hàng Nhật'),
    ('partner', 'Korean', 'f_n_b', 'Korean restaurant', 'Nhà hàng Hàn'),
    ('google', 'korean_restaurant', 'f_n_b', 'Korean restaurant', 'Nhà hàng Hàn'),
    ('partner', 'Thai', 'f_n_b', 'Thai restaurant', 'Nhà hàng Thái'),
    ('google', 'thai_restaurant', 'f_n_b', 'Thai restaurant', 'Nhà hàng Thái'),
    ('partner', 'French', 'f_n_b', 'French restaurant', 'Nhà hàng Pháp'),
    ('google', 'french_restaurant', 'f_n_b', 'French restaurant', 'Nhà hàng Pháp'),
    ('partner', 'Italian', 'f_n_b', 'Italian restaurant', 'Nhà hàng Ý'),
    ('google', 'italian_restaurant', 'f_n_b', 'Italian restaurant', 'Nhà hàng Ý'),
    ('google', 'pizza_restaurant', 'f_n_b', 'Italian restaurant', 'Nhà hàng Ý'),
    ('partner', 'American', 'f_n_b', 'American restaurant', 'Nhà hàng Mỹ'),
    ('google', 'american_restaurant', 'f_n_b', 'American restaurant', 'Nhà hàng Mỹ'),
    ('partner', 'Western', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'european_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'british_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'german_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'spanish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'greek_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'steak_house', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'portuguese_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'austrian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'bavarian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'belgian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'danish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'dutch_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'scandinavian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'swiss_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'polish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'romanian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'russian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'ukrainian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'croatian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'czech_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'hungarian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'australian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'californian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'tex_mex_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'fish_and_chips_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'basque_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'irish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'soul_food_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'southwestern_us_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('google', 'eastern_european_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu'),
    ('partner', 'International', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'indian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'north_indian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'south_indian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'indonesian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'malaysian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'filipino_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'cambodian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'burmese_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'tibetan_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'taiwanese_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'afghani_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'african_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'ethiopian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'moroccan_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'lebanese_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'israeli_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'persian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'turkish_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'middle_eastern_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'mediterranean_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'mexican_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'latin_american_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'south_american_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'brazilian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'argentinian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'colombian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'chilean_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'peruvian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'cuban_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'caribbean_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'hawaiian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'asian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'asian_fusion_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'mongolian_barbecue_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'pakistani_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'bangladeshi_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('google', 'sri_lankan_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế'),
    ('partner', 'Fusion', 'f_n_b', 'Fusion', 'Ẩm thực Fusion'),
    ('google', 'fusion_restaurant', 'f_n_b', 'Fusion', 'Ẩm thực Fusion'),
    ('google', 'bistro', 'f_n_b', 'Fusion', 'Ẩm thực Fusion'),
    ('partner', 'Bakery & Dessert', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng'),
    ('google', 'bakery', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng'),
    ('google', 'cake_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng'),
    ('google', 'pastry_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng'),
    ('google', 'dessert_restaurant', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng'),
    ('google', 'dessert_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng'),
    ('google', 'donut_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng'),
    ('google', 'ice_cream_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng'),
    ('google', 'confectionery', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng'),
    ('google', 'chocolate_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng'),
    ('partner', 'Bar & Pub/Nightlife', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('google', 'bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('google', 'pub', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('google', 'night_club', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('google', 'cocktail_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('google', 'wine_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('google', 'lounge_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('google', 'sports_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('google', 'hookah_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('google', 'irish_pub', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('google', 'beer_garden', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('google', 'karaoke', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife'),
    ('partner', 'Fast Food', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'fast_food_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'meal_takeaway', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'sandwich_shop', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'deli', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'snack_bar', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'burrito_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'taco_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'chicken_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'kebab_shop', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'gyro_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'shawarma_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('google', 'hamburger_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh'),
    ('partner', 'Vegan Restaurant', 'f_n_b', 'Vegan Restaurant', 'Nhà hàng Chay'),
    ('google', 'vegan_restaurant', 'f_n_b', 'Vegan Restaurant', 'Nhà hàng Chay'),
    ('google', 'vegetarian_restaurant', 'f_n_b', 'Vegan Restaurant', 'Nhà hàng Chay'),
    ('partner', 'Seafood Restaurant', 'f_n_b', 'Seafood Restaurant', 'Nhà hàng Hải sản'),
    ('google', 'seafood_restaurant', 'f_n_b', 'Seafood Restaurant', 'Nhà hàng Hải sản'),
    ('google', 'oyster_bar_restaurant', 'f_n_b', 'Seafood Restaurant', 'Nhà hàng Hải sản'),
    ('partner', 'Café', 'f_n_b', 'Café', 'Cà phê'),
    ('google', 'coffee_shop', 'f_n_b', 'Café', 'Cà phê'),
    ('google', 'cafe', 'f_n_b', 'Café', 'Cà phê'),
    ('google', 'coffee_roastery', 'f_n_b', 'Café', 'Cà phê'),
    ('google', 'coffee_stand', 'f_n_b', 'Café', 'Cà phê'),
    ('google', 'tea_house', 'f_n_b', 'Café', 'Cà phê'),
    ('google', 'juice_shop', 'f_n_b', 'Café', 'Cà phê'),
    ('google', 'hot_pot_restaurant', 'f_n_b', 'Hot Pot', 'Lẩu'),
    ('google', 'barbecue_restaurant', 'f_n_b', 'Barbecue & Grill', 'Nướng & BBQ'),
    ('google', 'bar_and_grill', 'f_n_b', 'Barbecue & Grill', 'Nướng & BBQ'),
    ('google', 'korean_barbecue_restaurant', 'f_n_b', 'Barbecue & Grill', 'Nướng & BBQ'),
    ('google', 'noodle_shop', 'f_n_b', 'Noodle', 'Mì & Món nước'),
    ('google', 'ramen_restaurant', 'f_n_b', 'Noodle', 'Mì & Món nước'),
    ('google', 'chinese_noodle_restaurant', 'f_n_b', 'Noodle', 'Mì & Món nước'),
    ('google', 'fine_dining_restaurant', 'f_n_b', 'Fine Dining', 'Fine Dining'),
    ('google', 'buffet_restaurant', 'f_n_b', 'Buffet', 'Buffet'),
    ('google', 'brunch_restaurant', 'f_n_b', 'Brunch & Breakfast', 'Bữa sáng & Brunch'),
    ('google', 'breakfast_restaurant', 'f_n_b', 'Brunch & Breakfast', 'Bữa sáng & Brunch'),
    ('google', 'halal_restaurant', 'f_n_b', 'Halal', 'Nhà hàng Halal'),
    ('google', 'gastropub', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery'),
    ('google', 'brewpub', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery'),
    ('google', 'brewery', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery'),
    ('google', 'winery', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery'),
    ('google', 'restaurant', 'f_n_b', 'F&B', 'Ẩm thực'),
    ('google', 'food_court', 'f_n_b', 'F&B', 'Ẩm thực'),
    ('google', 'cafeteria', 'f_n_b', 'F&B', 'Ẩm thực'),
    ('google', 'diner', 'f_n_b', 'F&B', 'Ẩm thực'),
    ('partner', 'Day Spa', 'spa_n_wellness', 'Spa', 'Spa'),
    ('google', 'spa', 'spa_n_wellness', 'Spa', 'Spa'),
    ('google', 'day_spa', 'spa_n_wellness', 'Spa', 'Spa'),
    ('google', 'wellness_center', 'spa_n_wellness', 'Spa', 'Spa'),
    ('partner', 'Therapeutic Massage', 'spa_n_wellness', 'Therapeutic Massage', 'Massage trị liệu'),
    ('google', 'massage', 'spa_n_wellness', 'Therapeutic Massage', 'Massage trị liệu'),
    ('partner', 'Clinic-Aesthetic Spa', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ'),
    ('google', 'skin_care_clinic', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ'),
    ('google', 'acupuncture_clinic', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ'),
    ('google', 'ayurvedic_clinic', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ'),
    ('partner', 'Blind Massage', 'spa_n_wellness', 'Blind Massage', 'Massage người khiếm thị'),
    ('partner', 'Onsen-Jjimjilbang', 'spa_n_wellness', 'Onsen-Jjimjilbang', 'Tắm Onsen – Xông hơi'),
    ('google', 'hot_spring', 'spa_n_wellness', 'Onsen-Jjimjilbang', 'Tắm Onsen – Xông hơi'),
    ('google', 'sauna', 'spa_n_wellness', 'Onsen-Jjimjilbang', 'Tắm Onsen – Xông hơi'),
    ('partner', 'Hair Salon', 'spa_n_wellness', 'Hair Salon', 'Salon'),
    ('google', 'hair_salon', 'spa_n_wellness', 'Hair Salon', 'Salon'),
    ('google', 'hair_care', 'spa_n_wellness', 'Hair Salon', 'Salon'),
    ('partner', 'Nail-Lash Salon', 'spa_n_wellness', 'Nail-Lash Salon', 'Tiệm Nail – Mi'),
    ('google', 'nail_salon', 'spa_n_wellness', 'Nail-Lash Salon', 'Tiệm Nail – Mi'),
    ('partner', 'Gym-Fitness', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình'),
    ('google', 'gym', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình'),
    ('google', 'fitness_center', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình'),
    ('google', 'sports_club', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình'),
    ('google', 'recreation_center', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình'),
    ('partner', 'Yoga-Pilates Studio', 'spa_n_wellness', 'Yoga-Pilates Studio', 'Phòng tập Yoga – Pilates'),
    ('google', 'yoga_studio', 'spa_n_wellness', 'Yoga-Pilates Studio', 'Phòng tập Yoga – Pilates'),
    ('google', 'pilates_studio', 'spa_n_wellness', 'Yoga-Pilates Studio', 'Phòng tập Yoga – Pilates'),
    ('partner', 'Meditation Center', 'spa_n_wellness', 'Meditation Center', 'Trung tâm Thiền'),
    ('google', 'meditation_center', 'spa_n_wellness', 'Meditation Center', 'Trung tâm Thiền'),
    ('google', 'beauty_salon', 'spa_n_wellness', 'Beauty Salon', 'Spa & Làm đẹp'),
    ('google', 'hair_removal_service', 'spa_n_wellness', 'Hair Removal Service', 'Dịch vụ triệt lông'),
    ('google', 'tanning_studio', 'spa_n_wellness', 'Tanning Studio', 'Dịch vụ nhuộm da'),
    ('google', 'barber_shop', 'spa_n_wellness', 'Barbershop', 'Barbershop'),
    ('google', 'body_piercing_shop', 'spa_n_wellness', 'Tattoo & Piercing', 'Xăm hình & Xỏ khuyên'),
    ('google', 'tattoo_parlor', 'spa_n_wellness', 'Tattoo & Piercing', 'Xăm hình & Xỏ khuyên'),
    ('google', 'swimming_pool', 'spa_n_wellness', 'Swimming Pool', 'Bể bơi'),
    ('partner', 'Local Specialty', 'shopping_n_retail', 'Local Specialty', 'Đặc sản địa phương'),
    ('partner', 'Souvenirs & Handicrafts', 'shopping_n_retail', 'Souvenirs & Handicrafts', 'Quà lưu niệm & Thủ công mỹ nghệ'),
    ('google', 'gift_shop', 'shopping_n_retail', 'Souvenirs & Handicrafts', 'Quà lưu niệm & Thủ công mỹ nghệ'),
    ('google', 'antique_shop', 'shopping_n_retail', 'Souvenirs & Handicrafts', 'Quà lưu niệm & Thủ công mỹ nghệ'),
    ('partner', 'Fashion & Accessories', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện'),
    ('google', 'clothing_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện'),
    ('google', 'shoe_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện'),
    ('google', 'fashion_accessories_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện'),
    ('google', 'hat_shop', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện'),
    ('google', 'sunglasses_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện'),
    ('google', 'swimwear_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện'),
    ('google', 'leather_goods_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện'),
    ('google', 'bag_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện'),
    ('partner', 'Jewelry & Gems', 'shopping_n_retail', 'Jewelry & Gems', 'Trang sức & Đá quý'),
    ('google', 'jewelry_store', 'shopping_n_retail', 'Jewelry & Gems', 'Trang sức & Đá quý'),
    ('partner', 'Cosmetics & Beauty', 'shopping_n_retail', 'Cosmetics & Beauty', 'Mỹ phẩm & Làm đẹp'),
    ('google', 'cosmetics_store', 'shopping_n_retail', 'Cosmetics & Beauty', 'Mỹ phẩm & Làm đẹp'),
    ('google', 'perfume_store', 'shopping_n_retail', 'Cosmetics & Beauty', 'Mỹ phẩm & Làm đẹp'),
    ('google', 'grocery_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('google', 'food_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('google', 'liquor_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('google', 'home_goods_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('google', 'furniture_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('google', 'home_improvement_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('google', 'hardware_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('google', 'book_store', 'shopping_n_retail', 'Books & Stationery', 'Sách – Văn phòng phẩm'),
    ('google', 'stationery_store', 'shopping_n_retail', 'Books & Stationery', 'Sách – Văn phòng phẩm'),
    ('google', 'electronics_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('google', 'cell_phone_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('google', 'shopping_mall', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị'),
    ('google', 'supermarket', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị'),
    ('google', 'department_store', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị'),
    ('google', 'discount_store', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị'),
    ('google', 'wholesaler', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị'),
    ('partner', 'Convenience Store', 'shopping_n_retail', 'Convenience Store', 'Cửa hàng tiện lợi'),
    ('google', 'convenience_store', 'shopping_n_retail', 'Convenience Store', 'Cửa hàng tiện lợi'),
    ('partner', 'Local/Night Market', 'shopping_n_retail', 'Local/ Night Market', 'Chợ truyền thống / Chợ đêm'),
    ('google', 'market', 'shopping_n_retail', 'Local/ Night Market', 'Chợ truyền thống / Chợ đêm'),
    ('partner', 'Sporting Goods', 'shopping_n_retail', 'Sporting Goods', 'Dụng cụ thể thao'),
    ('google', 'sporting_goods_store', 'shopping_n_retail', 'Sporting Goods', 'Dụng cụ thể thao'),
    ('google', 'bicycle_store', 'shopping_n_retail', 'Sporting Goods', 'Dụng cụ thể thao'),
    ('google', 'toy_store', 'shopping_n_retail', 'Toy Store', 'Cửa hàng đồ chơi'),
    ('google', 'baby_goods_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('google', 'luggage_store', 'shopping_n_retail', 'Luggage Store', 'Vali & Phụ kiện du lịch'),
    ('google', 'second_hand_shop', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('google', 'florist', 'shopping_n_retail', 'Florist', 'Cửa hàng hoa'),
    ('google', 'store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm'),
    ('partner', 'Home & Decor', 'shopping_n_retail', 'Home & Decor', 'Đồ gia dụng - Gốm sứ'),
    ('partner', 'Theme Park', 'attraction', 'Theme Park', 'Công viên giải trí'),
    ('google', 'amusement_park', 'attraction', 'Theme Park', 'Công viên giải trí'),
    ('google', 'amusement_center', 'attraction', 'Theme Park', 'Công viên giải trí'),
    ('google', 'fairground', 'attraction', 'Theme Park', 'Công viên giải trí'),
    ('partner', 'Water Park', 'attraction', 'Water Park', 'Công viên nước'),
    ('google', 'water_park', 'attraction', 'Water Park', 'Công viên nước'),
    ('partner', 'Museum', 'attraction', 'Museum', 'Bảo tàng'),
    ('google', 'museum', 'attraction', 'Museum', 'Bảo tàng'),
    ('google', 'planetarium', 'attraction', 'Museum', 'Bảo tàng'),
    ('partner', 'Historical Site', 'attraction', 'Historical Site', 'Di tích lịch sử'),
    ('google', 'historical_landmark', 'attraction', 'Historical Site', 'Di tích lịch sử'),
    ('google', 'archaeological_site', 'attraction', 'Historical Site', 'Di tích lịch sử'),
    ('google', 'ruins', 'attraction', 'Historical Site', 'Di tích lịch sử'),
    ('google', 'castle', 'attraction', 'Historical Site', 'Di tích lịch sử'),
    ('google', 'palace', 'attraction', 'Historical Site', 'Di tích lịch sử'),
    ('google', 'monument', 'attraction', 'Historical Site', 'Di tích lịch sử'),
    ('google', 'sculpture', 'attraction', 'Historical Site', 'Di tích lịch sử'),
    ('google', 'lighthouse', 'attraction', 'Historical Site', 'Di tích lịch sử'),
    ('partner', 'Zoo-Aquarium', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung'),
    ('google', 'zoo', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung'),
    ('google', 'aquarium', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung'),
    ('google', 'wildlife_park', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung'),
    ('partner', 'Nature Reserve', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên'),
    ('google', 'nature_reserve', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên'),
    ('google', 'national_park', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên'),
    ('google', 'hiking_area', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên'),
    ('google', 'wildlife_refuge', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên'),
    ('partner', 'Architecture-Landmark', 'attraction', 'Architecture-Landmark', 'Công trình kiến trúc'),
    ('google', 'cultural_landmark', 'attraction', 'Architecture-Landmark', 'Công trình kiến trúc'),
    ('partner', 'Art Show-Performance', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật'),
    ('google', 'concert_hall', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật'),
    ('google', 'performing_arts_theater', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật'),
    ('google', 'opera_house', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật'),
    ('google', 'comedy_club', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật'),
    ('google', 'amphitheatre', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật'),
    ('partner', 'Traditional Village', 'attraction', 'Traditional Village', 'Làng nghề'),
    ('partner', 'Religious Site', 'attraction', 'Religious Site', 'Điểm đến tâm linh'),
    ('google', 'church', 'attraction', 'Religious Site', 'Điểm đến tâm linh'),
    ('google', 'hindu_temple', 'attraction', 'Religious Site', 'Điểm đến tâm linh'),
    ('google', 'mosque', 'attraction', 'Religious Site', 'Điểm đến tâm linh'),
    ('google', 'synagogue', 'attraction', 'Religious Site', 'Điểm đến tâm linh'),
    ('google', 'buddhist_temple', 'attraction', 'Religious Site', 'Điểm đến tâm linh'),
    ('google', 'place_of_worship', 'attraction', 'Religious Site', 'Điểm đến tâm linh'),
    ('partner', 'Casino', 'attraction', 'Casino', 'Sòng bạc'),
    ('google', 'casino', 'attraction', 'Casino', 'Sòng bạc'),
    ('partner', 'Art Gallery', 'attraction', 'Art Gallery', 'Phòng tranh/ Xưởng nghệ thuật'),
    ('google', 'art_gallery', 'attraction', 'Art Gallery', 'Phòng tranh/ Xưởng nghệ thuật'),
    ('google', 'cultural_center', 'attraction', 'Cultural Center', 'Trung tâm văn hóa'),
    ('google', 'community_center', 'attraction', 'Cultural Center', 'Trung tâm văn hóa'),
    ('google', 'escape_room', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà'),
    ('google', 'video_arcade', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà'),
    ('google', 'bowling_alley', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà'),
    ('google', 'miniature_golf_course', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà'),
    ('google', 'go_kart_track', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà'),
    ('google', 'beach', 'attraction', 'Beach', 'Biển'),
    ('google', 'park', 'attraction', 'Park', 'Công viên'),
    ('google', 'viewpoint', 'attraction', 'Viewpoint', 'Điểm ngắm cảnh'),
    ('google', 'observation_deck', 'attraction', 'Observation Deck', 'Đài quan sát'),
    ('google', 'waterfall', 'attraction', 'Waterfall', 'Thác nước'),
    ('google', 'cave', 'attraction', 'Cave', 'Hang động'),
    ('google', 'botanical_garden', 'attraction', 'Botanical Garden', 'Vườn bách thảo'),
    ('google', 'stadium', 'attraction', 'Stadium', 'Sân vận động'),
    ('partner', 'Motorbike Rental', 'vehicle_rental', 'Motorbike Rental', 'Cho thuê xe máy'),
    ('google', 'scooter_rental', 'vehicle_rental', 'Motorbike Rental', 'Cho thuê xe máy'),
    ('partner', 'Self-drive Car Rental', 'vehicle_rental', 'Self-drive Car Rental', 'Cho thuê ô tô tự lái'),
    ('google', 'car_rental', 'vehicle_rental', 'Self-drive Car Rental', 'Cho thuê ô tô tự lái'),
    ('partner', 'Car with Driver', 'vehicle_rental', None, None),
    ('partner', 'Bicycle-EV Rental', 'vehicle_rental', 'Bicycle-EV Rental', 'Cho thuê xe đạp – Xe điện'),
    ('google', 'bicycle_rental', 'vehicle_rental', 'Bicycle-EV Rental', 'Cho thuê xe đạp – Xe điện'),
    ('partner', 'Campervan Rental', 'vehicle_rental', None, 'Cho thuê xe nhà di động'),
    ('google', 'boat_rental', 'vehicle_rental', 'Boat Rental', 'Cho thuê thuyền'),
    ('google', 'airport', 'other', 'airport', None),
    ('google', 'international_airport', 'other', 'airport', None),
    ('google', 'bus_station', 'other', 'bus_station', None),
    ('google', 'train_station', 'other', 'train_station', None),
    ('google', 'subway_station', 'other', 'metro_station', None),
    ('google', 'light_rail_station', 'other', 'metro_station', None),
    ('google', 'ferry_terminal', 'other', 'ferry_terminal', None),
    ('google', 'transit_station', 'other', 'transit_station', None),
    ('google', 'natural_feature', 'other', 'natural_feature', None),
    ('google', 'town_square', 'other', 'town_square', None),
    ('google', 'city_hall', 'other', 'government_offices', None),
    ('google', 'local_government_office', 'other', 'government_offices', None),
    ('google', 'embassy', 'other', 'embassy', None),
    ('google', 'police', 'other', 'police_and_fire_station', None),
    ('google', 'fire_station', 'other', 'police_and_fire_station', None),
    ('google', 'post_office', 'other', 'post_office', None),
    ('google', 'library', 'other', 'public_library', None),
    ('google', 'hospital', 'other', 'hospital', None),
    ('google', 'drugstore', 'other', 'drugstore', None),
    ('google', 'university', 'other', 'academic_building', None),
    ('google', 'school', 'other', 'academic_building', None),
]


CATEGORY_OTHER = "Other"
CATEGORY_GROUPS = (
    ("Lodging", ("lodging", "hotel", "resort_hotel", "motel", "guest_house", "hostel", "bed_and_breakfast", "resort")),
    ("Main Meals", ("restaurant", "food", "meal_takeaway", "seafood_restaurant", "bistro")),
    ("Light Snacks / Cafe", ("cafe", "bakery", "coffee_shop", "tea_house")),
    ("Heavy Entertainment", ("amusement_park", "theme_park", "water_park", "zoo", "aquarium", "national_park",
                             "hiking_area")),
    ("Light Sightseeing / Culture & Nature", ("tourist_attraction", "museum", "place_of_worship", "beach", "park",
                                              "historical_landmark", "pagoda", "church")),
    ("Shopping & Nightlife", ("shopping_mall", "market", "night_market", "bar", "night_club", "souvenir_store")),
)


def category_group(place_type):
    """Nhóm theo CATEGORY_GROUPS, khớp đúng chữ. Ví dụ: hotel -> Lodging ; Hotel (partner) -> Other."""
    for group, types in CATEGORY_GROUPS:
        if place_type in types:
            return group
    return CATEGORY_OTHER


def with_category_group(rows):
    """Dòng seed 5 trường -> 6 trường (thêm category_group)."""
    return [tuple(row) + (category_group(row[1]),) for row in rows]


def validate_seed(rows):
    """Khóa trùng, source, sector, placeholder, token google."""
    seen_token = {}
    for group, types in CATEGORY_GROUPS:
        for token in types:
            if token in seen_token:
                raise ValueError(f"token {token!r} in {seen_token[token]!r} and {group!r}")
            seen_token[token] = group
    seen = set()
    token_re = re.compile(r"^[a-z0-9_]+$")
    allowed = set(SEED_SECTORS)
    skip = set(SEED_SKIP)
    for source, place_type, sector, _category, _category_vi in rows:
        key = (source, place_type)
        if key in seen:
            raise ValueError(f"duplicate key {key}")
        seen.add(key)
        if source not in {"google", "partner"}:
            raise ValueError(f"source {source!r}")
        if sector not in allowed:
            raise ValueError(f"sector {sector!r}")
        if place_type in skip:
            raise ValueError(f"place_type {place_type!r}")
        if source == "google" and token_re.fullmatch(place_type) is None:
            raise ValueError(f"bad token {place_type!r}")
# END_PLACE_TYPE_SEED


In [ ]:
import json
from collections import Counter

REF_FQ = "lh_vv_silver.dbo.ref_dictionary_place_type"
SEED_SCHEMA = (
    "source_name string, place_type string, business_sector string, "
    "business_category string, business_category_vi string, category_group string"
)
apply = to_bool(p_apply)
validate_seed(SEED_ROWS)
rows6 = with_category_group(SEED_ROWS)
counts = Counter(row[0] for row in SEED_ROWS)
seed_df = spark.createDataFrame(rows6, SEED_SCHEMA)
seed_df.createOrReplaceTempView("seed_place_type")
if table_exists(REF_FQ):
    missing_keys = spark.sql(f"""
        SELECT count(*) AS n
        FROM seed_place_type s
        LEFT ANTI JOIN {REF_FQ} t
          ON t.source_name = s.source_name AND t.place_type = s.place_type
    """).first()["n"]
else:
    missing_keys = 0
planned_edges = [
    "slv_3p_poi", "slv_3p_poi_address", "ref_dictionary_place_type",
    "slv_pn_business_services", "slv_pn_business_service_i18ns",
]
planned_retire = ["slv_3p_poi", "ref_dictionary_place_type"]
print(f"seed google={counts['google']} partner={counts['partner']} missing_keys={missing_keys}")
print(f"planned edges: {len(planned_edges)} UPSERT -> slv_poi_entity_non_enrich; retire {len(planned_retire)} -> slv_poi_entity")
if not apply:
    notebookutils.notebook.exit(json.dumps({
        "status": "PLAN",
        "p_apply": False,
        "google": counts["google"],
        "partner": counts["partner"],
        "missing_keys": int(missing_keys),
        "planned_edges": planned_edges,
        "planned_retire": planned_retire,
    }))


In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lh_vv_silver.dbo.ref_dictionary_place_type (
    source_name          STRING NOT NULL,
    place_type           STRING NOT NULL,
    business_sector      STRING,
    business_category    STRING,
    business_category_vi STRING,
    is_active            BOOLEAN,
    updated_at           TIMESTAMP,
    category_group       STRING
) USING DELTA
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true', 'delta.autoOptimize.autoCompact' = 'true')
""")
EXPECTED = [
    ("source_name", "string", False),
    ("place_type", "string", False),
    ("business_sector", "string", True),
    ("business_category", "string", True),
    ("business_category_vi", "string", True),
    ("is_active", "boolean", True),
    ("updated_at", "timestamp", True),
    ("category_group", "string", True),
]
described = spark.table(REF_FQ).schema.fields
got = [(field.name.lower(), field.dataType.simpleString(), field.nullable) for field in described]
if got != EXPECTED:
    raise ConfigError(f"{REF_FQ} lệch cột hoặc nullability: có {got}, cần {EXPECTED}. Thiếu category_group: chạy cell [THÊM 19/10] của NB_CREATE_DDL")


In [ ]:
def _seed_merge():
    spark.sql("""
        MERGE INTO lh_vv_silver.dbo.ref_dictionary_place_type AS t
        USING seed_place_type AS s
        ON t.source_name = s.source_name AND t.place_type = s.place_type
        WHEN NOT MATCHED THEN INSERT
            (source_name, place_type, business_sector, business_category, business_category_vi, is_active, updated_at, category_group)
        VALUES (s.source_name, s.place_type, s.business_sector, s.business_category, s.business_category_vi,
                true, current_timestamp(), s.category_group)
    """)


missing_before = spark.sql("""
    SELECT count(*) AS n
    FROM seed_place_type s
    LEFT ANTI JOIN lh_vv_silver.dbo.ref_dictionary_place_type t
      ON t.source_name = s.source_name AND t.place_type = s.place_type
""").first()["n"]
with_retry(_seed_merge, "seed ref_dictionary_place_type")
INSERTED = int(missing_before)

dups = spark.sql("""
    SELECT count(*) AS n FROM (
        SELECT source_name, place_type
        FROM lh_vv_silver.dbo.ref_dictionary_place_type
        GROUP BY source_name, place_type
        HAVING count(*) > 1
    )
""").first()["n"]
if dups:
    raise ConfigError(f"ref_dictionary_place_type: {dups} khoá (source_name, place_type) trùng")

missing_after = spark.sql("""
    SELECT count(*) AS n
    FROM seed_place_type s
    LEFT ANTI JOIN lh_vv_silver.dbo.ref_dictionary_place_type t
      ON t.source_name = s.source_name AND t.place_type = s.place_type
""").first()["n"]
if missing_after:
    raise ConfigError(f"ref_dictionary_place_type: thiếu {missing_after} khoá seed")

DRIFT = int(spark.sql("""
    SELECT count(*) AS n
    FROM seed_place_type s
    JOIN lh_vv_silver.dbo.ref_dictionary_place_type t
      ON t.source_name = s.source_name AND t.place_type = s.place_type
    WHERE NOT (
        t.business_sector <=> s.business_sector
        AND t.business_category <=> s.business_category
        AND t.business_category_vi <=> s.business_category_vi
        AND t.category_group <=> s.category_group
        AND t.is_active = true
    )
""").first()["n"])
print(f"inserted={INSERTED} drift={DRIFT}")


In [ ]:
import uuid

# BEGIN_EDGE_PRECHECK
def setup_edge_issues(rows, upserts, retires):
    """Cạnh active trên lh_vv_silver.dbo mà MERGE không ghi thì chặn. Đúng bộ RETIRE thì cho qua.

    rows: pl_name, src_schema, src_tbl, trg_schema, trg_tbl, is_active.
    upserts, retires: tập (src_tbl, trg_tbl) của MERGE này, schema lh_vv_silver.dbo, pl SLV_1H.
    Schema khác không tính. Output: list thông báo, rỗng nghĩa là được MERGE.
    """
    silver = "lh_vv_silver.dbo"
    flow = "PL_VV_TRANS_SLV_TO_SLV_1H"
    targets = {trg for _src, trg in upserts}
    extra, stale_entity, stale_amenity = [], 0, 0
    for row in rows:
        if int(row.get("is_active") or 0) != 1:
            continue
        if row.get("src_schema") != silver or row.get("trg_schema") != silver:
            continue
        key = (row["src_tbl"], row["trg_tbl"])
        planned_retire = row.get("pl_name") == flow and key in retires
        if row["trg_tbl"] in targets and key not in upserts and not planned_retire:
            extra.append(f"{row['trg_tbl']}/{row['pl_name']}/{row['src_tbl']}")
        if row["trg_tbl"] == "slv_poi_entity" and not planned_retire:
            stale_entity += 1
        if key == ("slv_poi_entity", "slv_poi_amenity") and row.get("pl_name") != flow:
            stale_amenity += 1
    issues = []
    if extra:
        issues.append("còn cạnh active ngoài cạnh chuẩn: " + ", ".join(extra))
    if stale_entity:
        issues.append(f"slv_poi_entity: còn {stale_entity} cạnh active")
    if stale_amenity:
        issues.append("slv_poi_amenity: còn cạnh active slv_poi_entity")
    return issues
# END_EDGE_PRECHECK


def _write_edges():
    """Ghi cạnh entity và amenity trong một MERGE. Kiểm tra dưới lock, trước khi commit."""
    cfg = "lh_vv_ctrl.dbo.ctrl_mng_pipeline_config"
    has_nb = "nb_name" in table_columns(cfg)
    nb_match = " AND t.nb_name <=> CAST(NULL AS STRING)" if has_nb else ""
    nb_set = ", nb_name = NULL" if has_nb else ""
    nb_col = ", nb_name" if has_nb else ""
    nb_val = ", CAST(NULL AS STRING)" if has_nb else ""
    edge_rows = """
            (1, 'lh_vv_silver.dbo', 'slv_3p_poi', 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'UPSERT'),
            (2, 'lh_vv_silver.dbo', 'slv_3p_poi_address', 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'UPSERT'),
            (3, 'lh_vv_silver.dbo', 'ref_dictionary_place_type', 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'UPSERT'),
            (4, 'lh_vv_silver.dbo', 'slv_pn_business_services', 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'UPSERT'),
            (5, 'lh_vv_silver.dbo', 'slv_pn_business_service_i18ns', 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'UPSERT'),
            (0, 'lh_vv_silver.dbo', 'slv_3p_poi', 'lh_vv_silver.dbo', 'slv_poi_entity', 'RETIRE'),
            (0, 'lh_vv_silver.dbo', 'ref_dictionary_place_type', 'lh_vv_silver.dbo', 'slv_poi_entity', 'RETIRE'),
            (6, 'lh_vv_silver.dbo', 'slv_3p_poi_amenity', 'lh_vv_silver.dbo', 'slv_poi_amenity', 'UPSERT'),
            (7, 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'lh_vv_silver.dbo', 'slv_poi_amenity', 'UPSERT'),
            (8, 'lh_vv_silver.dbo', 'ref_dictionary_amenities', 'lh_vv_silver.dbo', 'slv_poi_amenity', 'UPSERT'),
            (0, 'lh_vv_silver.dbo', 'slv_poi_entity', 'lh_vv_silver.dbo', 'slv_poi_amenity', 'RETIRE')"""
    dup = spark.sql(f"""
        SELECT trg_tbl, src_tbl, count(*) AS n
        FROM {cfg}
        WHERE pl_name = 'PL_VV_TRANS_SLV_TO_SLV_1H'
          AND src_schema = 'lh_vv_silver.dbo'
          AND trg_schema = 'lh_vv_silver.dbo'
          AND (
            (trg_tbl = 'slv_poi_entity_non_enrich' AND src_tbl IN (
                'slv_3p_poi', 'slv_3p_poi_address', 'ref_dictionary_place_type',
                'slv_pn_business_services', 'slv_pn_business_service_i18ns'))
            OR (trg_tbl = 'slv_poi_entity' AND src_tbl IN ('slv_3p_poi', 'ref_dictionary_place_type'))
            OR (trg_tbl = 'slv_poi_amenity' AND src_tbl IN (
                'slv_3p_poi_amenity', 'slv_poi_entity_non_enrich', 'ref_dictionary_amenities', 'slv_poi_entity'))
          )
        GROUP BY src_schema, src_tbl, trg_schema, trg_tbl
        HAVING count(*) > 1
    """).collect()
    if dup:
        names = ", ".join(f"{row.trg_tbl}/{row.src_tbl}={row.n}" for row in dup)
        raise ConfigError(f"slv_poi_entity_non_enrich: trùng cạnh, không ghi đè; cạnh active không đúng: {names}")
    owners = spark.sql(f"""
        SELECT DISTINCT trg_tbl, pl_name
        FROM {cfg}
        WHERE trg_schema = 'lh_vv_silver.dbo'
          AND trg_tbl IN ('slv_poi_entity_non_enrich', 'slv_poi_amenity')
          AND is_active = 1
          AND pl_name <> 'PL_VV_TRANS_SLV_TO_SLV_1H'
    """).collect()
    if owners:
        names = ", ".join(f"{row.trg_tbl}/{row.pl_name}" for row in owners)
        raise ConfigError(f"slv_poi_entity_non_enrich đã thuộc pl_name khác: {names}")
    watched = spark.sql(f"""
        SELECT pl_name, src_schema, src_tbl, trg_schema, trg_tbl, is_active
        FROM {cfg}
        WHERE src_schema = 'lh_vv_silver.dbo' AND trg_schema = 'lh_vv_silver.dbo'
          AND (
            trg_tbl IN ('slv_poi_entity_non_enrich', 'slv_poi_amenity', 'slv_poi_entity')
            OR src_tbl = 'slv_poi_entity'
          )
    """).collect()
    entity_upserts = {
        ("slv_3p_poi", "slv_poi_entity_non_enrich"),
        ("slv_3p_poi_address", "slv_poi_entity_non_enrich"),
        ("ref_dictionary_place_type", "slv_poi_entity_non_enrich"),
        ("slv_pn_business_services", "slv_poi_entity_non_enrich"),
        ("slv_pn_business_service_i18ns", "slv_poi_entity_non_enrich"),
        ("slv_3p_poi_amenity", "slv_poi_amenity"),
        ("slv_poi_entity_non_enrich", "slv_poi_amenity"),
        ("ref_dictionary_amenities", "slv_poi_amenity"),
    }
    entity_retires = {
        ("slv_3p_poi", "slv_poi_entity"),
        ("ref_dictionary_place_type", "slv_poi_entity"),
        ("slv_poi_entity", "slv_poi_amenity"),
    }
    issues = setup_edge_issues([row.asDict() for row in watched], entity_upserts, entity_retires)
    if issues:
        raise ConfigError("; ".join(issues))

    missing = spark.sql(f"""
        SELECT count(*) AS n
        FROM VALUES {edge_rows} AS t(seq, src_schema, src_tbl, trg_schema, trg_tbl, action)
        WHERE t.action = 'UPSERT' AND NOT EXISTS (
            SELECT 1 FROM {cfg} c
            WHERE c.pl_name = 'PL_VV_TRANS_SLV_TO_SLV_1H'
              AND c.src_schema = t.src_schema AND c.src_tbl = t.src_tbl
              AND c.trg_schema = t.trg_schema AND c.trg_tbl = t.trg_tbl)
    """).first()["n"]
    retired = spark.sql(f"""
        SELECT count(*) AS n FROM {cfg}
        WHERE pl_name = 'PL_VV_TRANS_SLV_TO_SLV_1H'
          AND src_schema = 'lh_vv_silver.dbo'
          AND trg_schema = 'lh_vv_silver.dbo'
          AND (
            (trg_tbl = 'slv_poi_entity' AND src_tbl = 'slv_3p_poi' AND NOT (is_active <=> 0))
            OR (trg_tbl = 'slv_poi_entity' AND src_tbl = 'ref_dictionary_place_type' AND NOT (is_active <=> 0))
            OR (trg_tbl = 'slv_poi_amenity' AND src_tbl = 'slv_poi_entity' AND NOT (is_active <=> 0))
          )
    """).first()["n"]
    spark.sql(f"""
        MERGE INTO {cfg} AS t
        USING (
            SELECT seq, src_schema, src_tbl, trg_schema, trg_tbl, action
            FROM VALUES {edge_rows} AS v(seq, src_schema, src_tbl, trg_schema, trg_tbl, action)
        ) AS s
        ON t.pl_name = 'PL_VV_TRANS_SLV_TO_SLV_1H'
           AND t.src_schema = s.src_schema
           AND t.src_tbl = s.src_tbl
           AND t.trg_schema = s.trg_schema
           AND t.trg_tbl = s.trg_tbl
        WHEN MATCHED AND s.action = 'RETIRE' AND NOT (t.is_active <=> 0) THEN UPDATE SET
            is_active = 0
        WHEN MATCHED AND s.action = 'UPSERT' AND NOT (
            t.is_active <=> 1 AND t.priority <=> 1 AND t.load_mode <=> 'RECOMPUTE'
            AND t.src_schema <=> 'lh_vv_silver.dbo' AND t.trg_schema <=> 'lh_vv_silver.dbo'
            AND t.align_path <=> CAST(NULL AS STRING) AND t.dedup_order <=> CAST(NULL AS STRING){nb_match}
        ) THEN UPDATE SET
            src_schema = s.src_schema,
            trg_schema = s.trg_schema,
            is_active = 1,
            priority = 1,
            load_mode = 'RECOMPUTE',
            align_path = NULL,
            dedup_order = NULL{nb_set}
        WHEN NOT MATCHED AND s.action = 'UPSERT' THEN INSERT (
            id, pl_name, src_schema, src_tbl, trg_schema, trg_tbl,
            is_active, priority, load_mode, align_path, dedup_order{nb_col}
        ) VALUES (
            COALESCE((SELECT MAX(id) FROM {cfg}), 0) + s.seq,
            'PL_VV_TRANS_SLV_TO_SLV_1H', s.src_schema, s.src_tbl, s.trg_schema, s.trg_tbl,
            1, 1, 'RECOMPUTE', CAST(NULL AS STRING), CAST(NULL AS STRING){nb_val}
        )
    """)
    return int(missing), int(retired)


PL = "PL_VV_TRANS_SLV_TO_SLV_1H"
SETUP_EXEC = str(uuid.uuid4())
ensure_lock_row(PL, current_notebook_name("NB_SETUP_POI_ENTITY"))
ok, who = acquire_flow_lock(PL, SETUP_EXEC, SETUP_EXEC, LOCK_TIMEOUT_DEFAULT_MIN)
if not ok:
    raise ConfigError(f"Luồng {PL} đang chạy ({who}); chạy setup sau")
status = "FAILED"
try:
    EDGES_INSERTED, EDGES_RETIRED = with_retry(_write_edges, "edges slv_poi_entity_non_enrich")
    status = "SUCCESS"
    print(f"edges_inserted={EDGES_INSERTED} edges_retired={EDGES_RETIRED}")
finally:
    release_flow_lock(PL, SETUP_EXEC, status)


In [ ]:
import json

summary = {
    "status": "SUCCESS",
    "inserted": INSERTED,
    "drift": int(DRIFT),
    "edges_inserted": EDGES_INSERTED,
    "edges_retired": EDGES_RETIRED,
}
print(json.dumps(summary))
notebookutils.notebook.exit(json.dumps(summary))
